# Full Dataset Feature Build

This notebook creates a local optimized feature copy of OpenMementos.

The Hugging Face dataset is streamed once, parsed into trace-level and block-level feature tables, and saved as local parquet files under `data/`. The `data/` directory is ignored by Git, so these generated files are not committed.

The output is partitioned into multiple parquet files to avoid holding the full dataset in memory.

In [32]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.features import (
    DATASET_ID,
    SPLIT,
    ENCODING_NAME,
    write_feature_partitions,
)

# Number of original traces processed before writing one parquet partition.
CHUNK_SIZE = 10_000

# Use a small value for testing, then set to None for the full dataset.
#MAX_ROWS = 1_000
MAX_ROWS = None

In [11]:
# Create a run-specific output directory so full builds do not overwrite earlier runs.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("../data/full_feature_builds") / RUN_ID
TRACE_DIR = OUTPUT_DIR / "traces"
BLOCK_DIR = OUTPUT_DIR / "blocks"

TRACE_DIR.mkdir(parents=True, exist_ok=True)
BLOCK_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

PosixPath('../data/full_feature_builds/20260612_222038')

In [17]:
build_summary = write_feature_partitions(max_rows=MAX_ROWS)
build_summary

Streaming traces: 228557it [2:13:58, 28.43it/s] 


{'output_dir': '../data/full_feature_builds/20260612_222038',
 'n_trace_rows': 228557,
 'n_block_rows': 2013510,
 'n_parts': 23}

In [ ]:
# stream the dataset and write trace/block parquet partitions in OUTPUT_DIR
# run this section only to regenerate the full local parquet feature files

build_summary = write_feature_partitions(
    output_dir=OUTPUT_DIR,
    chunk_size=CHUNK_SIZE,
    max_rows=MAX_ROWS,
)

build_summary

In [18]:
df_traces_full_test = pd.read_parquet(TRACE_DIR)
df_blocks_full_test = pd.read_parquet(BLOCK_DIR)

df_traces_full_test.shape, df_blocks_full_test.shape

((228557, 19), (2013510, 21))

In [10]:
MAX_ROWS = None
# rerun from block #2

In [19]:
# df blocks 
df_blocks_full = pd.read_parquet(BLOCK_DIR)

# Define the high-compression target globally on the full local feature build.
compression_threshold = df_blocks_full["summary_to_block_token_ratio"].quantile(0.25)

df_blocks_full["high_token_compression"] = (
    df_blocks_full["summary_to_block_token_ratio"] <= compression_threshold
).astype(int)

compression_threshold, df_blocks_full["high_token_compression"].value_counts(normalize=True).round(4)

(np.float64(0.10891089108910891),
 high_token_compression
 0    0.75
 1    0.25
 Name: proportion, dtype: float64)

In [20]:
# Save a labeled block-level table for modeling notebooks.
df_blocks_full.to_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet", index=False)

In [24]:
# df traces
df_traces_full = pd.read_parquet(TRACE_DIR)

# Aggregate block-level compression behavior to the trace level. This creates
# one compression summary row per original reasoning trace.
trace_compression = (
    df_blocks_full
    .groupby("trace_id")
    .agg(
        trace_block_tokens=("block_tokens", "sum"),
        trace_summary_tokens=("summary_tokens", "sum"),
        trace_mean_summary_to_block_token_ratio=("summary_to_block_token_ratio", "mean"),
        trace_median_summary_to_block_token_ratio=("summary_to_block_token_ratio", "median"),
        trace_high_compression_share=("high_token_compression", "mean"),
    )
    .reset_index()
)

# The total trace compression ratio compares all summary tokens against all
# original block tokens within the same trace.
trace_compression["trace_total_summary_to_block_token_ratio"] = (
    trace_compression["trace_summary_tokens"]
    / trace_compression["trace_block_tokens"]
)

trace_compression.head()

,trace_id,trace_block_tokens,trace_summary_tokens,trace_mean_summary_to_block_token_ratio,trace_median_summary_to_block_token_ratio,trace_high_compression_share,trace_total_summary_to_block_token_ratio
0,0,3267,898,0.297850,0.303896,0.00,0.274870
1,1,2110,712,0.335386,0.270494,0.00,0.337441
2,2,14102,982,0.132226,0.056338,0.75,0.069636
3,3,5021,733,0.205201,0.104587,0.60,0.145987
4,4,3305,1077,0.350994,0.274430,0.00,0.325870


In [25]:
# Merge trace-level compression summaries onto the original trace-level table.
df_traces_full = df_traces_full.merge(
    trace_compression,
    on="trace_id",
    how="left",
)

df_traces_full.shape

(228557, 25)

In [26]:
# Define a trace-level high-compression target using the lowest quartile of the
# total trace compression ratio.
trace_compression_threshold = (
    df_traces_full["trace_total_summary_to_block_token_ratio"].quantile(0.25)
)

df_traces_full["trace_high_token_compression"] = (
    df_traces_full["trace_total_summary_to_block_token_ratio"] <= trace_compression_threshold
).astype(int)

trace_compression_threshold, df_traces_full["trace_high_token_compression"].value_counts(normalize=True).round(4)

(np.float64(0.1327609995400889),
 trace_high_token_compression
 0    0.75
 1    0.25
 Name: proportion, dtype: float64)

In [27]:
# Save labeled feature tables for downstream full-data modeling notebooks.
df_blocks_full.to_parquet(
    OUTPUT_DIR / "blocks_features_full_labeled.parquet",
    index=False,
)

df_traces_full.to_parquet(
    OUTPUT_DIR / "traces_features_full_labeled.parquet",
    index=False,
)

In [28]:
pd.read_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet").shape

(2013510, 22)

In [29]:
pd.read_parquet(OUTPUT_DIR / "traces_features_full_labeled.parquet").shape

(228557, 26)

## Full Feature Build Outputs

The full OpenMementos feature build produced two local labeled parquet files:

- `blocks_features_full_labeled.parquet`: block-level features and the block-level `high_token_compression` target.
- `traces_features_full_labeled.parquet`: trace-level features and the trace-level `trace_high_token_compression` target.

The block-level table has one row per `(reasoning block, summary)` pair. The trace-level table has one row per original reasoning trace.

These files are saved under the ignored `data/` directory and are not committed to Git.